# Few-Shot PCB Defect Segmentation Pipeline

Presentation notebook for the frozen arXiv-evidence protocol. Core logic lives in `src/` and `scripts/`; this notebook only loads compact evidence and selected assets.

## 1. Research Question and Disclosure

Research question: can anomaly-guided SAM2 prompting improve few-shot PCB defect segmentation over calibrated DINOv2 anomaly masks, especially for small or thin defects?

Prior local smoke tests use the synthetic/color-patch fixture only. Paper claims must come from the frozen VisA PCB primary and ablation matrices, not from smoke outputs or test-optimal oracle thresholds.

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
EVIDENCE_DIR = PROJECT_ROOT / "docs" / "evidence" / "generated"
ASSET_DIR = PROJECT_ROOT / "artifacts" / "paper_assets"


def read_csv(path: Path) -> list[dict[str, str]]:
    if not path.is_file():
        return []
    with path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def read_json(path: Path) -> dict:
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

## 2. Dataset and Repeated Few-Shot Protocol

VisA PCB categories `pcb1` to `pcb4` are the primary segmentation benchmark. Each primary method is evaluated for `k in {1, 2, 4}` normal supports and seeds `4880` to `4884` on fold 0. DeepPCB remains secondary because it has boxes rather than segmentation masks.

In [ ]:
manifest = read_json(EVIDENCE_DIR / "completion_manifest.json")
print("completion manifest present:", bool(manifest))
print("source runs:", len(manifest.get("source_runs", [])))

## 3. Method Summary

The frozen comparison includes PatchCore, single-scale DINOv2, multi-scale DINOv2, SAM2-only, single-scale DINOv2+SAM2, multi-scale DINOv2+SAM2, and anomaly-consistent SAM2 fusion. DINO/PatchCore heatmaps use normal-validation calibration; SAM2/fused methods produce binary masks.

## 4. Calibration Versus Oracle Diagnostics

Primary paper tables use calibrated binary masks (`normal_q995`). Oracle/test-optimal pixel thresholds are retained only in `oracle_diagnostics.csv` for diagnosis and must not be mixed into the main comparison table.

In [ ]:
oracle_rows = read_csv(EVIDENCE_DIR / "oracle_diagnostics.csv")
print("oracle diagnostic rows:", len(oracle_rows))
oracle_rows[:2]

## 5. Primary Results Table

Load compact paper evidence after the full AutoDL matrix and evidence builder complete.

In [ ]:
primary_rows = read_csv(EVIDENCE_DIR / "primary_results.csv")
print("primary rows:", len(primary_rows))
primary_rows[:5]

## 6. Paired Statistics and Failure Strata

Paired intervals compare multi-scale anomaly masks against unconditional SAM2 and anomaly-consistent SAM2. Defect-area and thinness strata are derived from pooled anomalous rows; normal rows are labeled separately.

In [ ]:
paired_statistics = read_json(EVIDENCE_DIR / "paired_statistics.json")
paired_statistics

## 7. Selected Success and Failure Figures

Qualitative assets are selected deterministically from `per_image_failure_analysis.csv` and copied into `artifacts/paper_assets/qualitative/`. Missing required panels are fatal during curation.

In [ ]:
qual_manifest = read_csv(ASSET_DIR / "qualitative_manifest.csv")
print("qualitative examples:", len(qual_manifest))
qual_manifest[:4]

## 8. Limitations and Decision Rule

If anomaly-consistent SAM2 improves calibrated mean anomaly mask F1/IoU without unacceptable degradation, the report can claim a useful anomaly-guided prompting bridge. If it does not improve, the arXiv-style report should present an empirical negative result and analyze when SAM2 hurts.

## 9. Exact Reproduction Commands

See `README.md` for local smoke commands and `docs/autodl_data_setup.md` for AutoDL primary/ablation matrix, checker, analysis, asset, and evidence commands. The readiness status is tracked in `docs/arxiv_readiness_checklist.md`.